In [17]:
import rby1_sdk as rby
import time
import numpy as np
from matplotlib import pyplot as plt
import inspect

np.set_printoptions(suppress=True, precision=3, linewidth=300)  

# Connect & Power ON

In [20]:
# ROBOT_ADDRESS = "localhost:50051"           # For Docker Simulation
ROBOT_ADDRESS = "192.168.30.1:50051"        # For Real Robot

robot = rby.create_robot_a(ROBOT_ADDRESS)   # Robot Model is "A"
model = robot.model()
if not robot.connect():
    print("FAIL: Could not connect to robot")
    exit(1)
else:
    print("SUCCESS: Connected to the robot!!\n")
    
    print("Power ON the Robot...")
    if (robot.power_on(".*") == True):
        time.sleep(2)
        if (robot.servo_on(".*") == True):
            robot.reset_fault_control_manager()
            if (robot.enable_control_manager(unlimited_mode_enabled=True) == True):
                print("Successfully Power ON the Robot")
            else:
                print("### Control manager is NOT Activated ###")
        else:
            print("### Servo is NOT Activated ###")
    else:
        print("### Power is NOT Activated ###")

SUCCESS: Connected to the robot!!

Power ON the Robot...
Successfully Power ON the Robot


# Power OFF

In [33]:
robot.disable_control_manager()
robot.power_off(".*")  

True

# Forward & Backward Translation

In [21]:
def build_mobility_cmd(vx, vy=0.0, yaw_rate=0.0, hold_time=0.3, acc=0.2):
    return rby.RobotCommandBuilder().set_command(
        rby.ComponentBasedCommandBuilder().set_mobility_command(
            rby.SE2VelocityCommandBuilder()
            .set_velocity(np.array([vx, vy]), yaw_rate)
            .set_acceleration_limit(np.array([acc, acc]), acc)
            .set_command_header(
                rby.CommandHeaderBuilder().set_control_hold_time(hold_time)
            )
        )
    )


def move_forward_distance_smooth(
    robot,
    distance_m,
    max_speed=0.3,       # 0.3m/s
    accel=0.1,           # m/s^2, 작을수록 천천히 가속
    tolerance=0.01,
    dt=0.02              # 50 Hz
):
    robot.reset_odometry(
        angle=0.0,
        position=np.array([0.0, 0.0])
    )

    direction = 1.0 if distance_m >= 0 else -1.0
    target = abs(distance_m)
    current_speed = 0.0

    stream = robot.create_command_stream(priority=1)

    try:
        while True:
            odom = robot.get_state().odometry
            x = odom[0, 2]
            y = odom[1, 2]

            moved = np.sqrt(x*x + y*y)
            remain = target - moved

            if remain <= tolerance:
                break

            # 남은 거리에서 안전하게 멈출 수 있는 속도
            # v^2 = 2 a x
            brake_speed = np.sqrt(max(0.0, 2.0 * accel * remain))

            # 목표 속도: 최대속도와 제동 가능 속도 중 작은 값
            target_speed = min(max_speed, brake_speed)
            
            print(f"moved={moved:.3f}, speed={current_speed:.3f}")

            # 한 루프마다 속도 변화량 제한
            max_dv = accel * dt

            if target_speed > current_speed:
                current_speed = min(current_speed + max_dv, target_speed)
            else:
                current_speed = max(current_speed - max_dv, target_speed)

            vx = direction * current_speed

            cmd = build_mobility_cmd(
                vx,
                0.0,
                0.0,
                hold_time=0.3,
                acc=accel
            )

            # 핵심: .get()으로 매번 기다리지 않고 stream으로 연속 전송
            stream.send_command(cmd, timeout_ms=500)

            time.sleep(dt)

        # 마지막 감속 정지
        # (1) 일반적인 선형 감속
        # for _ in range(20):
        #     current_speed = max(0.0, current_speed - accel * dt)
        #     vx = direction * current_speed

        #     cmd = build_mobility_cmd(
        #         vx,
        #         0.0,
        #         0.0,
        #         hold_time=0.3,
        #         acc=accel
        #     )
        #     stream.send_command(cmd, timeout_ms=500)
        #     time.sleep(dt)
        
        # (2) 더 부드러운 ease-out
        stop_duration = 2.0  # 멈추는 데 걸리는 시간, 길수록 부드러움
        steps = int(stop_duration / dt)

        start_speed = current_speed

        for i in range(steps):
            t = (i + 1) / steps  # 0 → 1

            # smoothstep: 처음/끝 변화가 부드러움
            s = t * t * (3 - 2 * t)

            current_speed = start_speed * (1.0 - s)
            vx = direction * current_speed

            cmd = build_mobility_cmd(
                vx,
                0.0,
                0.0,
                hold_time=0.3,
                acc=accel
            )
            stream.send_command(cmd, timeout_ms=1000)
            time.sleep(dt)

    finally:
        # 완전 정지 명령 여러 번 보내기
        for _ in range(5):
            stream.send_command(
                build_mobility_cmd(0.0, 0.0, 0.0, hold_time=0.5, acc=accel),
                timeout_ms=1000
            )
            time.sleep(0.05)

        stream.cancel()

In [5]:
move_forward_distance_smooth(robot, 2.0)

moved=0.000, speed=0.000
moved=0.000, speed=0.002
moved=0.000, speed=0.004
moved=0.000, speed=0.006
moved=0.000, speed=0.008
moved=0.000, speed=0.010
moved=0.000, speed=0.012
moved=0.000, speed=0.014
moved=0.000, speed=0.016
moved=0.000, speed=0.018
moved=0.000, speed=0.020
moved=0.000, speed=0.022
moved=0.001, speed=0.024
moved=0.001, speed=0.026
moved=0.001, speed=0.028
moved=0.001, speed=0.030
moved=0.001, speed=0.032
moved=0.001, speed=0.034
moved=0.001, speed=0.036
moved=0.002, speed=0.038
moved=0.002, speed=0.040
moved=0.002, speed=0.042
moved=0.002, speed=0.044
moved=0.003, speed=0.046
moved=0.003, speed=0.048
moved=0.003, speed=0.050
moved=0.003, speed=0.052
moved=0.004, speed=0.054
moved=0.004, speed=0.056
moved=0.005, speed=0.058
moved=0.005, speed=0.060
moved=0.005, speed=0.062
moved=0.006, speed=0.064
moved=0.006, speed=0.066
moved=0.007, speed=0.068
moved=0.007, speed=0.070
moved=0.008, speed=0.072
moved=0.009, speed=0.074
moved=0.009, speed=0.076
moved=0.010, speed=0.078


# Rotation

In [30]:
def get_yaw(odom):
    return np.arctan2(odom[1, 0], odom[0, 0])

def normalize_angle(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi

def rotate_inplace_smooth(
    robot,
    target_deg=90,
    max_yaw_rate=0.1,
    min_yaw_rate=0.01,
    accel=0.02,
    tolerance_deg=1.0,
    dt=0.02
):
    robot.reset_odometry(angle=0.0, position=np.array([0.0, 0.0]))

    target_rad = np.deg2rad(target_deg)
    tolerance = np.deg2rad(tolerance_deg)

    current_rate = 0.0
    stream = robot.create_command_stream(priority=1)

    try:
        while True:
            odom = robot.get_state().odometry
            theta = get_yaw(odom)

            error = normalize_angle(target_rad - theta)

            print(
                f"theta_deg={np.rad2deg(theta):.2f}, "
                f"error_deg={np.rad2deg(error):.2f}, "
                f"rate={current_rate:.3f}"
            )

            if abs(error) < tolerance:
                break

            direction = np.sign(error)

            brake_rate = np.sqrt(max(0.0, 2.0 * accel * abs(error)))
            target_rate = min(max_yaw_rate, brake_rate)

            # 너무 작은 회전속도는 실제 로봇에서 안 먹을 수 있어서 최소값 부여
            target_rate = max(min_yaw_rate, target_rate)

            max_dr = accel * dt
            rate_mag = abs(current_rate)

            if target_rate > rate_mag:
                rate_mag = min(rate_mag + max_dr, target_rate)
            else:
                rate_mag = max(rate_mag - max_dr, target_rate)

            current_rate = direction * rate_mag
            yaw_rate = current_rate

            cmd = build_mobility_cmd(
                0.0,
                0.0,
                yaw_rate,
                hold_time=0.3,
                acc=accel
            )

            stream.send_command(cmd, timeout_ms=500)
            time.sleep(dt)

    finally:
        # 완전 정지
        for _ in range(5):
            stream.send_command(
                build_mobility_cmd(0.0, 0.0, 0.0, hold_time=0.5),
                timeout_ms=1000
            )
            time.sleep(0.05)

        stream.cancel()

In [32]:
rotate_inplace_smooth(robot, 90)    # 좌회전 90도
# rotate_inplace_smooth(robot, -90)   # 우회전 90도

theta_deg=0.00, error_deg=90.00, rate=0.000
theta_deg=0.00, error_deg=90.00, rate=0.000
theta_deg=0.00, error_deg=90.00, rate=0.001
theta_deg=0.00, error_deg=90.00, rate=0.001
theta_deg=0.00, error_deg=90.00, rate=0.002
theta_deg=0.00, error_deg=90.00, rate=0.002
theta_deg=0.00, error_deg=90.00, rate=0.002
theta_deg=0.00, error_deg=90.00, rate=0.003
theta_deg=0.00, error_deg=90.00, rate=0.003
theta_deg=0.00, error_deg=90.00, rate=0.004
theta_deg=0.00, error_deg=90.00, rate=0.004
theta_deg=0.00, error_deg=90.00, rate=0.004
theta_deg=0.00, error_deg=90.00, rate=0.005
theta_deg=0.00, error_deg=90.00, rate=0.005
theta_deg=0.00, error_deg=90.00, rate=0.006
theta_deg=0.00, error_deg=90.00, rate=0.006
theta_deg=0.01, error_deg=89.99, rate=0.006
theta_deg=0.01, error_deg=89.99, rate=0.007
theta_deg=0.01, error_deg=89.99, rate=0.007
theta_deg=0.01, error_deg=89.99, rate=0.008
theta_deg=0.01, error_deg=89.99, rate=0.008
theta_deg=0.01, error_deg=89.99, rate=0.008
theta_deg=0.01, error_deg=89.99,